# SAGB | ATP Visualizations — Clean Edition

Open this notebook in Jupyter and choose **Run → Run All Cells**. The figures will appear and be saved as PNG and SVG files. You only need this notebook and the two accompanying CSV files.

Use the **STYLE CONTROLS** cell to change highlighted players, text colors, or the heatmap's `SHOW_CELL_VALUES` setting. The underlying data and percentages stay unchanged.

# ATP Tennis — Article Figures
Clean, publication-oriented versions of your PCA map and upset-rate heatmap.
Data is unchanged; only presentation has been redesigned.
In Jupyter, run all cells. Change the STYLE CONTROLS below and rerun for revisions.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from pathlib import Path

base = Path('.')
profiles = pd.read_csv(base / 'ATP_Player_Profiles_2014_2025.csv')
matchups = pd.read_csv(base / 'ATP_Graph2_Heatmap_Results_2023_2025.csv')
print(f"Loaded {len(profiles):,} player profiles and {len(matchups)} matchup groups")

## STYLE CONTROLS
SHOW_CELL_VALUES=False matches your Displ ayr reference: red-gradient tiles + colorbar.
Set SHOW_CELL_VALUES=True to show percentages in the 3×3 matrix.

In [ ]:
INK = '#22343A'
MUTED = '#66747B'
GRID = '#DFE5E5'
ACCENT = '#BA4A34'
FIGURE_BG = '#FFFFFF'
SHOW_CELL_VALUES = False

HIGHLIGHT_PLAYERS = [
    'Ivo Karlovic', 'John Isner', 'Milos Raonic', 'Nick Kyrgios',
    'Roger Federer', 'Novak Djokovic', 'Rafael Nadal', 'Diego Schwartzman'
]

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'figure.facecolor': FIGURE_BG,
    'axes.facecolor': FIGURE_BG,
    'savefig.facecolor': FIGURE_BG,
    'text.color': INK,
    'axes.labelcolor': INK,
    'xtick.color': MUTED,
    'ytick.color': MUTED,
    'svg.fonttype': 'none',
    'savefig.dpi': 300,
})

## Graph 1 — ATP player PCA map
No caption below the plot; your article paragraphs can supply the interpretation.
Low-match players remain included (shown as smaller/lighter points).

In [ ]:
fig, ax = plt.subplots(figsize=(12.5, 7.0))
fig.subplots_adjust(left=.12, right=.96, bottom=.145, top=.76)
fig.text(.12, .925, 'How ATP players differ on serve and return',
         fontsize=22, fontweight='bold', ha='left')
fig.text(.12, .865, 'PCA of seven match-statistic measures  |  2014–2025',
         color=MUTED, fontsize=11, ha='left')

is_highlight = profiles['player_name'].isin(HIGHLIGHT_PLAYERS)
low_count = profiles['matches'] < 3
for sel, size, opacity in [
    (~is_highlight & low_count, 11, .18),
    (~is_highlight & ~low_count, 21, .47),
]:
    ax.scatter(profiles.loc[sel, 'PC1'], profiles.loc[sel, 'PC2'],
               s=size, color='#9FAEB3', alpha=opacity, linewidths=0, zorder=2)

ax.axhline(0, color=GRID, lw=.9, zorder=0)
ax.axvline(0, color=GRID, lw=.9, zorder=0)

# Hand-tuned offsets in display points, not data units
label_offsets = {
    'Ivo Karlovic': (10, 12),
    'John Isner': (10, -13),
    'Milos Raonic': (-68, 14),
    'Nick Kyrgios': (-70, 12),
    'Roger Federer': (9, -10),
    'Novak Djokovic': (10, -12),
    'Rafael Nadal': (-65, -13),
    'Diego Schwartzman': (-100, 8),
}
for name in HIGHLIGHT_PLAYERS:
    row = profiles.loc[profiles['player_name'] == name]
    if row.empty:
        print('Skipping absent player:', name)
        continue
    p = row.iloc[0]
    ax.scatter([p.PC1], [p.PC2], s=76, color=ACCENT,
               edgecolors='white', linewidths=1.3, zorder=5)
    ax.annotate(name, (p.PC1, p.PC2), xytext=label_offsets.get(name, (8, 8)),
                textcoords='offset points', fontsize=9.1, color=INK,
                ha='left', va='center', zorder=6)

ax.set_xlim(-9.3, 6.7)
ax.set_ylim(-4.6, 9.1)
ax.set_xlabel('PC1   ·   Serving effectiveness  →', fontsize=11, labelpad=14)
ax.set_ylabel('PC2   ·   Serve risk vs. return consistency  ↑', fontsize=11, labelpad=13)
ax.spines[['top','right']].set_visible(False)
ax.spines[['left','bottom']].set_color(GRID)
ax.tick_params(length=0, pad=8, labelsize=9)
ax.grid(False)

fig.savefig('SAGB_Graph1_PCA_Clean.png', bbox_inches='tight', pad_inches=.18)
fig.savefig('SAGB_Graph1_PCA_Clean.svg', bbox_inches='tight', pad_inches=.18)
plt.show()

## Graph 2 — Red-gradient heatmap
No statistics footnote or commentary below. The red color scale shows the raw upset rate.
Matches the reference's touching tiles, subtle grid, and vertical legend.

In [ ]:
order = ['Low PC2', 'Middle PC2', 'High PC2']
pivot = matchups.pivot(index='underdog_PC2_group', columns='favorite_PC2_group',
                       values='upset_rate').loc[order, order]
values = pivot.to_numpy(dtype=float)

# Coral → red → burgundy, close to the example's visual language
reds = LinearSegmentedColormap.from_list('editorial_reds', [
    '#FFC5B2', '#FF8F76', '#F45245', '#D71D2B', '#740019'
])
scale = Normalize(vmin=.32, vmax=.43)

fig = plt.figure(figsize=(11.0, 6.6), facecolor=FIGURE_BG)
fig.text(.10, .94, 'Which matchups have the highest upset rates?',
         fontsize=22, fontweight='bold', ha='left', va='center')
fig.text(.10, .879, 'ATP lower-ranked player win rate  |  2023–2025',
         fontsize=11, color=MUTED, ha='left', va='center')

# Extra width and touching rectangular cells, like the supplied reference
ax = fig.add_axes([.21, .18, .59, .57])
im = ax.imshow(values, cmap=reds, norm=scale, interpolation='nearest', aspect='auto')
ax.set_xlim(-.5, 2.5)
ax.set_ylim(2.5, -.5)

ax.set_xticks(range(3), ['Low', 'Middle', 'High'])
ax.set_yticks(range(3), ['Low', 'Middle', 'High'])
ax.xaxis.tick_top()
ax.tick_params(axis='x', which='major', length=0, labelsize=11, pad=12)
ax.tick_params(axis='y', which='major', length=0, labelsize=11, pad=10)
ax.set_xlabel('HIGHER-RANKED PLAYER · PC2 GROUP', fontsize=10,
              color=MUTED, labelpad=15, fontweight='bold')
ax.xaxis.set_label_position('top')
ax.set_ylabel('LOWER-RANKED PLAYER · PC2 GROUP', fontsize=10,
              color=MUTED, labelpad=18, fontweight='bold')

# Continuous heatmap, faint dividing rules (not separate floating boxes)
ax.set_xticks(np.arange(-.5, 3, 1), minor=True)
ax.set_yticks(np.arange(-.5, 3, 1), minor=True)
ax.grid(which='minor', color='#FFFFFF', linewidth=1.4, alpha=.4)
ax.tick_params(which='minor', bottom=False, left=False)
for sp in ax.spines.values():
    sp.set_visible(False)

if SHOW_CELL_VALUES:
    for r in range(3):
        for c in range(3):
            val = values[r, c]
            color = '#FFFFFF' if val >= .387 else '#4D2120'
            ax.text(c, r, f'{val:.1%}', ha='center', va='center',
                    fontsize=16, weight='semibold', color=color)

# Thin, unobtrusive gradient legend on the right
cax = fig.add_axes([.835, .18, .029, .57])
cbar = fig.colorbar(im, cax=cax)
cbar.outline.set_visible(False)
cbar.set_ticks([.32, .34, .36, .38, .40, .42])
cbar.set_ticklabels(['32%', '34%', '36%', '38%', '40%', '42%'])
cbar.ax.tick_params(labelsize=9.5, length=0, pad=7, colors=MUTED)
cbar.set_label('UPSET RATE', rotation=270, labelpad=24,
               fontsize=10, fontweight='bold', color=MUTED)

fig.savefig('SAGB_Graph2_Heatmap_Red.png', bbox_inches='tight', pad_inches=.18)
fig.savefig('SAGB_Graph2_Heatmap_Red.svg', bbox_inches='tight', pad_inches=.18)
plt.show()

## Exports
Outputs are saved beside this notebook as **high-resolution PNG** and **editable SVG**.
The input data, match-count definitions, pre-match PCA group assignments, and
observed upset rates have not changed. Only graphic presentation was revised.